# CBRAIN OASIS Threshold and Calibration Analysis (Step 3C)

This notebook analyzes threshold trade-offs and probability calibration for the frozen Step 3A baseline model.

Important: threshold tuning done on validation is exploratory and should be locked only after an external holdout or nested CV protocol.

In [ ]:
# 1) Load baseline prediction artifacts
from __future__ import annotations

import csv
import json
from datetime import date
from pathlib import Path

import numpy as np

try:
    from sklearn.metrics import (
        accuracy_score,
        brier_score_loss,
        f1_score,
        log_loss,
        precision_score,
        recall_score,
    )
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError('scikit-learn is required for Step 3C. Use .venv/bin/python.') from exc

cwd = Path.cwd().resolve()
candidate_roots = [cwd]
if cwd.name == 'notebooks':
    candidate_roots.append(cwd.parent)
else:
    candidate_roots.append(cwd / 'projects' / 'cbrain_oasis_cognition')
    candidate_roots.append(cwd.parent)

repo_root = None
for candidate in candidate_roots:
    if (candidate / 'results' / 'modeling' / 'baseline_validation_predictions.csv').exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError('Could not locate baseline prediction artifacts.')

model_dir = repo_root / 'results' / 'modeling'
docs_path = repo_root / 'docs' / 'threshold_calibration_step3c.md'

val_pred_path = model_dir / 'baseline_validation_predictions.csv'
train_pred_path = model_dir / 'baseline_train_predictions.csv'
baseline_metrics_path = model_dir / 'baseline_metrics.json'

def _read_csv(path: Path) -> tuple[list[str], list[dict]]:
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        return list(reader.fieldnames or []), [dict(r) for r in reader]

val_cols, val_rows = _read_csv(val_pred_path)
train_cols, train_rows = _read_csv(train_pred_path)
baseline_metrics = json.loads(baseline_metrics_path.read_text(encoding='utf-8'))

print(f'Repo root: {repo_root}')
print(f'Validation rows: {len(val_rows)}; Train rows: {len(train_rows)}')

In [ ]:
# 2) Contract checks
required_cols = ['Subject.ID', 'y_true', 'y_probability', 'y_pred_threshold_0_5']
if val_cols != required_cols:
    raise AssertionError(f'Validation prediction columns mismatch: {val_cols}')
if train_cols != required_cols:
    raise AssertionError(f'Train prediction columns mismatch: {train_cols}')

def _extract(rows: list[dict], split_name: str):
    sids = []
    y_true = []
    y_prob = []
    for r in rows:
        sid = r['Subject.ID']
        yt = int(r['y_true'])
        yp = float(r['y_probability'])
        if yt not in (0, 1):
            raise AssertionError(f'Invalid y_true in {split_name} for {sid}: {yt}')
        if yp < 0.0 or yp > 1.0:
            raise AssertionError(f'Invalid probability in {split_name} for {sid}: {yp}')
        sids.append(sid)
        y_true.append(yt)
        y_prob.append(yp)
    if len(set(sids)) != len(sids):
        raise AssertionError(f'Duplicate Subject.ID in {split_name} predictions.')
    return np.array(y_true, dtype=int), np.array(y_prob, dtype=float), sids

y_val, p_val, val_sids = _extract(val_rows, 'validation')
y_train, p_train, train_sids = _extract(train_rows, 'train')

if baseline_metrics['validation_row_count'] != len(y_val):
    raise AssertionError('Validation row count mismatch vs baseline_metrics.json')

print('Prediction contracts: PASS')

In [ ]:
# 3) Threshold sweep on validation
def _metrics_at_threshold(y_true: np.ndarray, p: np.ndarray, thr: float) -> dict:
    y_pred = (p >= thr).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    recall = float(recall_score(y_true, y_pred, zero_division=0))
    precision = float(precision_score(y_true, y_pred, zero_division=0))
    accuracy = float(accuracy_score(y_true, y_pred))
    f1 = float(f1_score(y_true, y_pred, zero_division=0))
    specificity = float(tn / (tn + fp) if (tn + fp) else 0.0)
    balanced_accuracy = float((recall + specificity) / 2.0)
    youden_j = float(recall + specificity - 1.0)
    return {
        'threshold': float(thr),
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'f1': f1,
        'balanced_accuracy': balanced_accuracy,
        'youden_j': youden_j,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
    }

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)
threshold_rows = [_metrics_at_threshold(y_val, p_val, float(t)) for t in thresholds]

at_050 = next(r for r in threshold_rows if abs(r['threshold'] - 0.5) < 1e-12)
best_f1 = max(threshold_rows, key=lambda r: (r['f1'], -abs(r['threshold'] - 0.5)))
best_youden = max(threshold_rows, key=lambda r: (r['youden_j'], -abs(r['threshold'] - 0.5)))
best_balacc = max(threshold_rows, key=lambda r: (r['balanced_accuracy'], -abs(r['threshold'] - 0.5)))

print('Threshold=0.50 F1:', round(at_050['f1'],4), 'Recall:', round(at_050['recall'],4), 'Specificity:', round(at_050['specificity'],4))
print('Best F1 threshold:', best_f1['threshold'], 'F1:', round(best_f1['f1'],4))

In [ ]:
# 4) Calibration diagnostics (validation + train reference)
def _calibration_table(y_true: np.ndarray, p: np.ndarray, bins: int = 10) -> tuple[list[dict], float, float]:
    edges = np.linspace(0.0, 1.0, bins + 1)
    rows = []
    n_total = len(y_true)
    ece = 0.0
    mce = 0.0
    for i in range(bins):
        lo = float(edges[i])
        hi = float(edges[i + 1])
        if i < bins - 1:
            idx = np.where((p >= lo) & (p < hi))[0]
        else:
            idx = np.where((p >= lo) & (p <= hi))[0]
        cnt = int(len(idx))
        if cnt == 0:
            rows.append({
                'bin_left': lo, 'bin_right': hi, 'count': 0,
                'mean_predicted': None, 'observed_rate': None, 'abs_gap': None,
            })
            continue
        mean_p = float(np.mean(p[idx]))
        obs = float(np.mean(y_true[idx]))
        gap = abs(mean_p - obs)
        ece += (cnt / n_total) * gap
        mce = max(mce, gap)
        rows.append({
            'bin_left': lo, 'bin_right': hi, 'count': cnt,
            'mean_predicted': mean_p, 'observed_rate': obs, 'abs_gap': float(gap),
        })
    return rows, float(ece), float(mce)

cal_rows_val, ece_val, mce_val = _calibration_table(y_val, p_val, bins=10)
cal_rows_train, ece_train, mce_train = _calibration_table(y_train, p_train, bins=10)

cal_summary = {
    'validation': {
        'brier_score': float(brier_score_loss(y_val, p_val)),
        'log_loss': float(log_loss(y_val, p_val, labels=[0,1])),
        'ece_10_bins': ece_val,
        'mce_10_bins': mce_val,
    },
    'train_reference': {
        'brier_score': float(brier_score_loss(y_train, p_train)),
        'log_loss': float(log_loss(y_train, p_train, labels=[0,1])),
        'ece_10_bins': ece_train,
        'mce_10_bins': mce_train,
    },
}

print('Validation Brier:', round(cal_summary['validation']['brier_score'], 4))
print('Validation ECE  :', round(cal_summary['validation']['ece_10_bins'], 4))

In [ ]:
# 5) Materialize Step 3C outputs + documentation
threshold_csv = model_dir / "step3c_threshold_sweep.csv"
cal_bins_csv = model_dir / "step3c_calibration_bins.csv"
threshold_json = model_dir / "step3c_threshold_analysis.json"
cal_json = model_dir / "step3c_calibration_summary.json"

with threshold_csv.open('w', encoding='utf-8', newline='') as f:
    fields = ['threshold','accuracy','precision','recall','specificity','f1','balanced_accuracy','youden_j','tp','tn','fp','fn']
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    for row in threshold_rows:
        w.writerow(row)

with cal_bins_csv.open('w', encoding='utf-8', newline='') as f:
    fields = ['split','bin_left','bin_right','count','mean_predicted','observed_rate','abs_gap']
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    for r in cal_rows_val:
        out = dict(r)
        out['split'] = 'validation'
        w.writerow(out)
    for r in cal_rows_train:
        out = dict(r)
        out['split'] = 'train_reference'
        w.writerow(out)

threshold_summary = {
    'date': date.today().isoformat(),
    'model_name': baseline_metrics['model_name'],
    'validation_row_count': len(y_val),
    'threshold_at_0_50': at_050,
    'best_threshold_by_f1': best_f1,
    'best_threshold_by_youden_j': best_youden,
    'best_threshold_by_balanced_accuracy': best_balacc,
    'note': 'Threshold selection on validation is exploratory; do not lock clinical threshold without external holdout.',
}

threshold_json.write_text(json.dumps(threshold_summary, indent=2, sort_keys=True) + "\n", encoding='utf-8')
cal_json.write_text(json.dumps(cal_summary, indent=2, sort_keys=True) + "\n", encoding='utf-8')

run_date = date.today().isoformat()
docs_text = f"""# Threshold and Calibration Analysis (Step 3C)

Date: {run_date}  
Notebook: `notebooks/06_threshold_calibration_step3c.ipynb`

## Scope

Analyze threshold trade-offs and calibration for the frozen Step 3A baseline probabilities.

## Threshold Sweep (Validation)

- Checked thresholds from 0.05 to 0.95 (step 0.01).
- At threshold 0.50: F1={at_050["f1"]:.4f}, Recall={at_050["recall"]:.4f}, Specificity={at_050["specificity"]:.4f}.
- Best by F1: threshold={best_f1["threshold"]:.2f}, F1={best_f1["f1"]:.4f}.
- Best by Youden's J: threshold={best_youden["threshold"]:.2f}, J={best_youden["youden_j"]:.4f}.
- Best by Balanced Accuracy: threshold={best_balacc["threshold"]:.2f}, BA={best_balacc["balanced_accuracy"]:.4f}.

## Calibration Summary

- Validation Brier score: {cal_summary["validation"]["brier_score"]:.4f}
- Validation log loss: {cal_summary["validation"]["log_loss"]:.4f}
- Validation ECE (10 bins): {cal_summary["validation"]["ece_10_bins"]:.4f}
- Validation MCE (10 bins): {cal_summary["validation"]["mce_10_bins"]:.4f}

Train-reference (overfit context):
- Train Brier score: {cal_summary["train_reference"]["brier_score"]:.4f}
- Train ECE (10 bins): {cal_summary["train_reference"]["ece_10_bins"]:.4f}

## Interpretation Guardrail

Threshold optimization performed on validation should be treated as exploratory.
For a production or claim-level threshold, lock it via a separate holdout or nested CV protocol.

## Step 3C Artifacts

- `results/modeling/step3c_threshold_sweep.csv`
- `results/modeling/step3c_threshold_analysis.json`
- `results/modeling/step3c_calibration_bins.csv`
- `results/modeling/step3c_calibration_summary.json`
"""
docs_path.write_text(docs_text, encoding='utf-8')

print('Wrote:', threshold_csv)
print('Wrote:', threshold_json)
print('Wrote:', cal_bins_csv)
print('Wrote:', cal_json)
print('Wrote:', docs_path)